In [36]:
from langchain_classic.chains.question_answering.map_rerank_prompt import output_parser
# model
from langchain_ollama import ChatOllama

llm = ChatOllama(model="qwen2.5:7b", temperature=0.8)
print(llm)

model='qwen2.5:7b' temperature=0.8


Prompt template

In [37]:
# prompt template
from langchain_core.prompts import ChatPromptTemplate
template_string = """Translate the text
that is delimited by triple backslashes
into a style that is {style}.
text: '''{text}'''
"""

prompt_template = ChatPromptTemplate.from_template(template_string)

prompt = prompt_template.messages[0].prompt
print("prompt完整内容：\n", prompt)
print("prompt中需要包含的属性：", prompt.input_variables)

prompt完整内容：
 input_variables=['style', 'text'] input_types={} partial_variables={} template="Translate the text\nthat is delimited by triple backslashes\ninto a style that is {style}.\ntext: '''{text}'''\n"
prompt中需要包含的属性： ['style', 'text']


In [38]:
# style
customer_style = "American English in a calm friendly formal and respect tone, at last translate to Simplified Chinese"

# text
customer_email = """
Arrr, I be fuming that me blender lid \
flew off and splattered me kitchen walls \
with smoothie! And to make matters worse, \
the warranty don't cover the cost of \
cleaning up me kitchen. I need yer help \
right now, matey!
"""

# 传入缺少的style和text
customer_message = prompt_template.format_messages(
    style=customer_style,
    text=customer_email,
)

print(customer_message[0])

content="Translate the text\nthat is delimited by triple backslashes\ninto a style that is American English in a calm friendly formal and respect tone, at last translate to Simplified Chinese.\ntext: '''\nArrr, I be fuming that me blender lid flew off and splattered me kitchen walls with smoothie! And to make matters worse, the warranty don't cover the cost of cleaning up me kitchen. I need yer help right now, matey!\n'''\n" additional_kwargs={} response_metadata={}


In [40]:
# 输入LLM并返回
customer_response = llm.invoke(customer_message)
print(customer_response.content)

Certainly! Here's the translation into a calm, friendly, formal, and respectful American English tone:

"Ugh, I'm really upset because my blender lid flew off and splashed smoothie all over my kitchen walls! And to make things worse, the warranty doesn't cover the cost of cleaning it up. Could you please help me right away? Thanks a lot!"

Now, here is the translation into Simplified Chinese:

“啊，我真的生气了，因为搅拌机盖子飞开了，弄得到处都是 smoothie！而且更糟糕的是，保修不包括清洁厨房的费用。你能现在帮我一下吗？非常感谢！” 

请注意，“smoothie”在中文中通常指的是smoothie（smoothie），但在上述句子中为了保持原意未翻译，因为它是一个专有名词。如果需要进一步解释或翻译成中文的具体食品名称，请告知。


In [14]:
# 回复内容
service_reply = """保修不包括你的厨房的清洁费用，
因为你使用搅拌机的时候忘记盖上盖子了，这是你操作的失误，这是你的问题。真倒霉！再见！
"""

# 风格要求
service_style_reply = "a interesting tone that speak in English, and use some rhyme like a rap"

# prompt
service_message = prompt_template.format_messages(
    style=service_style_reply,
    text=service_reply,
)

print(service_message)

[HumanMessage(content="Translate the text\nthat is delimited by triple backslashes\ninto a style that is a interesting tone that speak in English, and use some rhyme like a rap.\ntext: '''保修不包括你的厨房的清洁费用，\n因为你使用搅拌机的时候忘记盖上盖子了，这是你操作的失误，这是你的问题。真倒霉！再见！\n'''\n", additional_kwargs={}, response_metadata={})]


In [15]:
# generation response
service_response = llm.invoke(service_message)
print(service_response)

Yo, listen up, here's the deal,
When it comes to warranties, we’re not your meal.
Your kitchen’s shine? Not on our list—
You forgot to cap that blender just a bit.

A miss is as good as a mile, my friend,
You used it wrong, you should've been smarter then.
Oopsie-daisy, you're outta luck today,
It's your mistake, no need to cry or play.

So bye-bye now, and hope you learn from this,
Take better care next time, we ain't your nurse.
Stay tuned for more rap wisdom, keep it real,
This is the truth, in case you didn’t feel.


Output parser

In [16]:
# 某用户的商品评价
customer_review = """
MacBook Pro特别棒！特别喜欢！m2处理器性能超强，就是价钱有点小贵！电池续航逆天！不发热！还带有黑科技触控栏！
现在Mac 软件还算蛮多的，常用的办公软件都能有！用来日常办公完全没问题！
我想重点点评一下他的音频接口！这代MacBook Pro 带有先进的高阻抗耳机支持功能！同样的耳机，
插MacBook Pro上，效果要好于iPhone！还有它的录音性能！插上一根简单的转接头后，在配合电容麦，
还有库乐队软件，录音效果逆天！真的特别棒！我有比较老版本的Mac，但这代MacBook Pro的录音效果，
真的比以前的Mac效果要好好多倍！特别逆天！适合音乐人！（个人感觉，不插电源，录音效果似乎会更好！）
"""

# prompt
review_template = """
For the following text, extract the following information:

gift: was the item purchased as a gift for someone else?
answer true if yes, false if not or unknown.

delivery_days: how many days did it take for the product to arrive?
If this information is not found, output -1.

price_value: extract any sentences about the value or price,
and output them as a Python list, seperated by comma.

Format the output as JSON with the following format:
gift
delivery_days
price_value

text: {text}
"""

In [41]:
from langchain_core.prompts import ChatPromptTemplate
prompt_template = ChatPromptTemplate.from_template(review_template)
messages = prompt_template.format_messages(
    text=customer_review
)
print(messages)

from langchain_ollama import ChatOllama
llm = ChatOllama(model="qwen2.5:7b", temperature=0.8)

response = llm.invoke(messages)
print("\n", response)

[HumanMessage(content='\nFor the following text, extract the following information:\n\ngift: was the item purchased as a gift for someone else?\nanswer true if yes, false if not or unknown.\n\ndelivery_days: how many days did it take for the product to arrive?\nIf this information is not found, output -1.\n\nprice_value: extract any sentences about the value or price,\nand output them as a Python list, seperated by comma.\n\nFormat the output as JSON with the following format:\ngift\ndelivery_days\nprice_value\n\ntext: \nMacBook Pro特别棒！特别喜欢！m2处理器性能超强，就是价钱有点小贵！电池续航逆天！不发热！还带有黑科技触控栏！\n现在Mac 软件还算蛮多的，常用的办公软件都能有！用来日常办公完全没问题！\n我想重点点评一下他的音频接口！这代MacBook Pro 带有先进的高阻抗耳机支持功能！同样的耳机，\n插MacBook Pro上，效果要好于iPhone！还有它的录音性能！插上一根简单的转接头后，在配合电容麦，\n还有库乐队软件，录音效果逆天！真的特别棒！我有比较老版本的Mac，但这代MacBook Pro的录音效果，\n真的比以前的Mac效果要好好多倍！特别逆天！适合音乐人！（个人感觉，不插电源，录音效果似乎会更好！）\n\n', additional_kwargs={}, response_metadata={})]

 content='```json\n{\n  "gift": false,\n  "delivery_days": -1,\n  "price_value": ["价钱有点小贵！", "录音效果逆天！真的特别棒

Parse the LLM output string into Python dictionary

In [48]:
from langchain_classic.output_parsers import ResponseSchema
from langchain_classic.output_parsers import StructuredOutputParser

# 详细描述每一个期望返回的内容
gift_schema = ResponseSchema(
    name="gift",
    description="was the item purchased as a gift for someone else? Answer true if yes, false if not or unknown.",
)
delivery_days_schema = ResponseSchema(
    name="delivery_days",
    description="how many days did it take for the product to arrive? If this information is not found, output -1.",
)
price_value_schema = ResponseSchema(
    name="price_value",
    description="extract any sentences about the value or price, and output them as a Python list, seperated by comma.",
)

# 以字典形式创建 schema
response_schema = [
    gift_schema,
    delivery_days_schema,
    price_value_schema,
]

# 创建 OutputParser
output_parser = StructuredOutputParser.from_response_schemas(response_schema)

# 生成指令，后续作为prompt的一部分
format_instructions = output_parser.get_format_instructions()
print(format_instructions)

The output should be a markdown code snippet formatted in the following schema, including the leading and trailing "```json" and "```":

```json
{
	"gift": string  // was the item purchased as a gift for someone else? Answer true if yes, false if not or unknown.
	"delivery_days": string  // how many days did it take for the product to arrive? If this information is not found, output -1.
	"price_value": string  // extract any sentences about the value or price, and output them as a Python list, seperated by comma.
}
```


In [49]:
review_template_2 = """
For the following text, extract the following information:

gift: was the item purchased as a gift for someone else? Answer true if yes, false if not or unknown.
delivery_days: how many days did it take for the product to arrive? If this information is not found, output -1.
price_value: extract any sentences about the value or price, and output them as a Python list.

text: {text}

{format_instructions}
"""

prompt_template = ChatPromptTemplate.from_template(review_template_2)
prompt = prompt_template.format_messages(
    text=customer_review,
    format_instructions=format_instructions,
)

print(prompt)

[HumanMessage(content='\nFor the following text, extract the following information:\n\ngift: was the item purchased as a gift for someone else? Answer true if yes, false if not or unknown.\ndelivery_days: how many days did it take for the product to arrive? If this information is not found, output -1.\nprice_value: extract any sentences about the value or price, and output them as a Python list.\n\ntext: \nMacBook Pro特别棒！特别喜欢！m2处理器性能超强，就是价钱有点小贵！电池续航逆天！不发热！还带有黑科技触控栏！\n现在Mac 软件还算蛮多的，常用的办公软件都能有！用来日常办公完全没问题！\n我想重点点评一下他的音频接口！这代MacBook Pro 带有先进的高阻抗耳机支持功能！同样的耳机，\n插MacBook Pro上，效果要好于iPhone！还有它的录音性能！插上一根简单的转接头后，在配合电容麦，\n还有库乐队软件，录音效果逆天！真的特别棒！我有比较老版本的Mac，但这代MacBook Pro的录音效果，\n真的比以前的Mac效果要好好多倍！特别逆天！适合音乐人！（个人感觉，不插电源，录音效果似乎会更好！）\n\n\nThe output should be a markdown code snippet formatted in the following schema, including the leading and trailing "```json" and "```":\n\n```json\n{\n\t"gift": string  // was the item purchased as a gift for someone else? Answer true if yes, false if not or unknown.\n\

In [50]:
response = llm.invoke(prompt)

# 将结果解析成对应的字典
output_dict = output_parser.parse(response.content)

print(output_dict.get('price_value'))

['价钱有点小贵！']
